# Métricas por Região + ULAS (S5.8) — Kaggle

**Autor**: Massanori  
**Data**: 21/05/2026 (Célula 7 corrigida em 22/05/2026)  
**Descrição**: Orquestra `scripts/compute_metrics.py` para os 3 grupos (A/B/C) sobre o split `test` (47 volumes anotados). Produz 3 CSVs por-slice + 3 JSONs sumário, contendo `Coverage_global`, `Coverage_lesion`, `mean_width_global`, `mean_width_lesion`, `IoU_topk_global`, `IoU_topk_lesion`, `ULAS_lesion` (contribuição original) + null baseline.

## Pré-requisitos

- Settings: GPU **T4 x1** se a cota estiver disponivel; em CPU funciona, leva ~15 min/grupo.
- Add Input (6 datasets):
  1. `tcc-mri-recons-varnet-brain-4x` — split `test` (S4)
  2. `tcc-mri-lesion-masks` — máscaras .pt (S3)
  3. `tcc-mri-resm-checkpoints` — best.pt do Grupo A
  4. `tcc-mri-qr-checkpoints` — best.pt do Grupo B
  5. `tcc-mri-qr-lesion-checkpoints` — best.pt do Grupo C
  6. `tcc-mri-conformal-qhats` — 3 JSONs do S5.7

## Saída

`tcc-mri-s5-8-metrics` (publicado na célula final):
- `metrics_A.csv`, `metrics_B.csv`, `metrics_C.csv` — uma linha por slice
- `metrics_A.summary.json`, `metrics_B.summary.json`, `metrics_C.summary.json` — stats agregadas

## Citações relevantes

- Romano et al. (2019, NeurIPS 32): garantia de cobertura marginal
- Clopper & Pearson (1934, Biometrika): IC exato para proporções (Coverage)
- Demsar (2006, JMLR 7): testes estatísticos para comparações múltiplas (S5.9)
- Sobel & Feldman (1968): operador de gradiente do ULAS

In [ ]:
# Célula 2 — Setup: clone repo + dependencias
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = 'https://github.com/KR0N0S7/tcc-mri-uncertainty.git'
REPO_DIR = Path('/kaggle/working/tcc-mri-uncertainty')

if REPO_DIR.is_dir():
    print(f'Repo ja existe em {REPO_DIR}; git pull...')
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only'], check=True)
else:
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)

os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

print('\nHEAD do repo:')
subprocess.run(['git', '-C', str(REPO_DIR), 'log', '-1', '--oneline'], check=False)

print('\nInstalando dependencias...')
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    '-r', str(REPO_DIR / 'requirements.txt'),
    'python-dotenv',
], check=True)
print('Setup OK')

In [ ]:
# Célula 3 — Diagnostico: GPU + localiza recons + masks + 3 checkpoints + 3 qhats
import os
import subprocess
import sys
from pathlib import Path

RECONS_SLUG = 'tcc-mri-recons-varnet-brain-4x'
MASKS_SLUG = 'tcc-mri-lesion-masks'
CKPT_SLUGS = {
    'A': 'tcc-mri-resm-checkpoints',
    'B': 'tcc-mri-qr-checkpoints',
    'C': 'tcc-mri-qr-lesion-checkpoints',
}
QHATS_SLUG = 'tcc-mri-conformal-qhats'


def kaggle_input_candidates(slug):
    candidates = [Path('/kaggle/input') / slug]
    datasets_root = Path('/kaggle/input/datasets')
    if datasets_root.is_dir():
        for user_dir in datasets_root.iterdir():
            if user_dir.is_dir():
                candidates.append(user_dir / slug)
    return candidates


def find_dataset_root(slug, validator):
    """validator(path) -> True se path eh o dir certo (com aninhamento 1)."""
    for cand in kaggle_input_candidates(slug):
        if not cand.is_dir():
            continue
        if validator(cand):
            return cand
        sub_dirs = [p for p in cand.iterdir() if p.is_dir()]
        if len(sub_dirs) == 1 and validator(sub_dirs[0]):
            return sub_dirs[0]
    raise FileNotFoundError(
        f'Dataset \"{slug}\" nao encontrado em /kaggle/input/. '
        f'Verifique Add Input no painel direito.'
    )


def validate_recons(p):
    return all((p / s).is_dir() for s in ('train', 'val', 'cal', 'test'))


def validate_masks(p):
    return any(p.glob('*.pt'))


def validate_checkpoint(p):
    return (p / 'best.pt').is_file()


def validate_qhats(p):
    expected = {'q_hat_A.json', 'q_hat_B.json', 'q_hat_C.json'}
    files = {f.name for f in p.iterdir() if f.is_file()}
    return expected.issubset(files)


print('=' * 60)
print('DIAGNOSTICO')
print('=' * 60)

# 1. GPU
import torch
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'\n[1] Device: {DEVICE}')
if DEVICE == 'cuda':
    print(f'    GPU: {torch.cuda.get_device_name(0)}')
    print(f'    VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('    Rodando em CPU — ~15 min/grupo (vs ~5 min em GPU).')
    print('    Tudo bem, ja temos fallback np.partition no quantile.')

# 2. Recons
print(f'\n[2] Recons:')
RECONS_ROOT = find_dataset_root(RECONS_SLUG, validate_recons)
print(f'    -> {RECONS_ROOT}')
n_test = len(list((RECONS_ROOT / 'test').glob('*.npz')))
print(f'    test: {n_test} arquivos .npz')
if n_test == 0:
    raise FileNotFoundError(f'Split test vazio em {RECONS_ROOT / "test"}')

# 3. Masks
print(f'\n[3] Masks:')
MASKS_ROOT = find_dataset_root(MASKS_SLUG, validate_masks)
n_masks = len(list(MASKS_ROOT.glob('*.pt')))
print(f'    -> {MASKS_ROOT}')
print(f'    {n_masks} arquivos .pt')

# 4. Checkpoints
print(f'\n[4] Checkpoints:')
CHECKPOINT_PATHS = {}
for group, slug in CKPT_SLUGS.items():
    ckpt_root = find_dataset_root(slug, validate_checkpoint)
    CHECKPOINT_PATHS[group] = ckpt_root / 'best.pt'
    size_mb = CHECKPOINT_PATHS[group].stat().st_size / 1e6
    print(f'    Grupo {group}: {ckpt_root.name} (best.pt={size_mb:.1f} MB)')

# 5. q_hats
print(f'\n[5] q_hats (do S5.7):')
QHATS_ROOT = find_dataset_root(QHATS_SLUG, validate_qhats)
print(f'    -> {QHATS_ROOT}')
QHAT_PATHS = {g: QHATS_ROOT / f'q_hat_{g}.json' for g in 'ABC'}
import json
for g, p in QHAT_PATHS.items():
    data = json.loads(p.read_text())
    print(f'    Grupo {g}: q_hat={data["q_hat"]:.6f}, '
          f'sha256={data["checkpoint_sha256"][:16]}...')

# 6. Env vars
os.environ['TCC_RECONS_DIR'] = str(RECONS_ROOT)
os.environ['TCC_ANOTADOS_DIR'] = '/kaggle/working/_dummy_anotados'
os.environ['TCC_BRAIN_CSV'] = '/kaggle/working/_dummy_brain.csv'
Path('/kaggle/working/_dummy_anotados').mkdir(exist_ok=True)
Path('/kaggle/working/_dummy_brain.csv').touch()

print('\n' + '=' * 60)
print('DIAGNOSTICO OK')
print('=' * 60)

In [ ]:
# Célula 4 — Computa metricas Grupo A (ResM via Scaled CP)
import subprocess
import sys

OUTPUT_A = '/kaggle/working/metrics_A.csv'

result = subprocess.run([
    sys.executable, 'scripts/compute_metrics.py',
    '--group', 'A',
    '--checkpoint', str(CHECKPOINT_PATHS['A']),
    '--qhat', str(QHAT_PATHS['A']),
    '--recons-dir', str(RECONS_ROOT),
    '--masks-dir', str(MASKS_ROOT),
    '--output', OUTPUT_A,
    '--device', DEVICE,
    '--top-pct', '0.05',
    '--n-perms-ulas', '10',
    '--log-every', '100',
])

assert result.returncode == 0, 'Compute metrics Grupo A falhou.'
print(f'\nmetrics_A salvo em {OUTPUT_A}')

In [ ]:
# Célula 5 — Computa metricas Grupo B (QR via CQR)
import subprocess
import sys

OUTPUT_B = '/kaggle/working/metrics_B.csv'

result = subprocess.run([
    sys.executable, 'scripts/compute_metrics.py',
    '--group', 'B',
    '--checkpoint', str(CHECKPOINT_PATHS['B']),
    '--qhat', str(QHAT_PATHS['B']),
    '--recons-dir', str(RECONS_ROOT),
    '--masks-dir', str(MASKS_ROOT),
    '--output', OUTPUT_B,
    '--device', DEVICE,
    '--top-pct', '0.05',
    '--n-perms-ulas', '10',
    '--log-every', '100',
])

assert result.returncode == 0, 'Compute metrics Grupo B falhou.'
print(f'\nmetrics_B salvo em {OUTPUT_B}')

In [ ]:
# Célula 6 — Computa metricas Grupo C (QR-Lesion via CQR)
import subprocess
import sys

OUTPUT_C = '/kaggle/working/metrics_C.csv'

result = subprocess.run([
    sys.executable, 'scripts/compute_metrics.py',
    '--group', 'C',
    '--checkpoint', str(CHECKPOINT_PATHS['C']),
    '--qhat', str(QHAT_PATHS['C']),
    '--recons-dir', str(RECONS_ROOT),
    '--masks-dir', str(MASKS_ROOT),
    '--output', OUTPUT_C,
    '--device', DEVICE,
    '--top-pct', '0.05',
    '--n-perms-ulas', '10',
    '--log-every', '100',
])

assert result.returncode == 0, 'Compute metrics Grupo C falhou.'
print(f'\nmetrics_C salvo em {OUTPUT_C}')

In [ ]:
# Célula 7 — Tabela comparativa A vs B vs C (sumario por grupo) [FIX 22/05/2026: largura]
# Bug anterior: fmt(...)[:21].ljust(22) truncava 'n=736' em 'n=73' e 'n=362' em 'n=36'.
# Fix: COL_WIDTH=29 acomoda ate '0.0000 ± 0.0000 (n=99999)' (24 chars) com folga.
import json
from pathlib import Path

summaries = {}
for g in 'ABC':
    p = Path(f'/kaggle/working/metrics_{g}.summary.json')
    summaries[g] = json.loads(p.read_text())

COL_WIDTH = 29
KEY_WIDTH = 24
TOTAL_WIDTH = KEY_WIDTH + 3 * COL_WIDTH

print('=' * TOTAL_WIDTH)
print(f'METRICAS S5.8 — Comparativo A vs B vs C (split test, alpha=0.10)')
print('=' * TOTAL_WIDTH)

# Headline por grupo: n slices + qhat + sha
for g in 'ABC':
    s = summaries[g]
    print(f'Grupo {g}: {s["n_slices_total"]} slices '
          f'({s["n_slices_with_lesion"]} com lesao), '
          f'q_hat={s["q_hat"]:.6f}, '
          f'sha256={s["checkpoint_sha256"][:8]}...')

print()

# Header da tabela
header = f'{"Metrica":<{KEY_WIDTH}}'
for label in ['A (ResM)', 'B (QR)', 'C (QR-Lesion)']:
    header += f'{label:<{COL_WIDTH}}'
print(header)
print('-' * TOTAL_WIDTH)


def fmt(s, key):
    m = s['metrics_summary'][key]
    return f'{m["mean"]:.4f} ± {m["std"]:.4f} (n={m["n_valid"]})'


for key in ['coverage_global', 'coverage_lesion',
            'mean_width_global', 'mean_width_lesion',
            'iou_topk_global', 'iou_topk_lesion',
            'ulas_lesion', 'ulas_null_mean', 'ulas_z_score']:
    row = f'{key:<{KEY_WIDTH}}'
    for g in 'ABC':
        row += f'{fmt(summaries[g], key):<{COL_WIDTH}}'
    print(row)

print('=' * TOTAL_WIDTH)
print()
print('Leitura (predicao a priori — ver S5.9 para resultados estatisticos):')
print('  - coverage_global ~ 0.90 nos 3 grupos: garantia formal de Romano et al. (2019).')
print('  - coverage_lesion: predicao C > B > A (lambda=5 ajuda lesoes).')
print('  - mean_width_lesion: C deve ser ligeiramente > B (intervalos mais conservadores).')
print('  - iou_topk_lesion: predicao C > B (alinha incerteza com erro nas lesoes).')
print('  - ulas_lesion: predicao C > B > A (contribuicao original do TCC).')
print('  - ulas_z_score: se >> 2.0, alinhamento estatisticamente significativo.')
print()
print('Proximo passo (S5.9): Friedman + Nemenyi sobre os CSVs para significancia.')
print('Para regenerar esta tabela offline a partir dos JSONs:')
print('  python scripts/print_metrics_table.py --summary-dir <dir>')

In [ ]:
# Célula 8 — Empacota e publica como Kaggle dataset tcc-mri-s5-8-metrics
import json
import shutil
import subprocess
from pathlib import Path

KAGGLE_USER = 'massanorikishi'
DATASET_SLUG = 'tcc-mri-s5-8-metrics'
DATASET_TITLE = 'TCC MRI S5.8 Metrics (per-slice CSV + summary)'
OUTPUT_DIR = Path('/kaggle/working/tcc-mri-s5-8-metrics')
OUTPUT_DIR.mkdir(exist_ok=True)

# Copia 6 arquivos (3 CSV + 3 JSON)
for g in 'ABC':
    for ext in ['.csv', '.summary.json']:
        src = Path(f'/kaggle/working/metrics_{g}{ext}')
        dst = OUTPUT_DIR / src.name
        shutil.copy2(src, dst)
        size_kb = src.stat().st_size / 1024
        print(f'  copiado: {src.name} ({size_kb:.1f} kB)')

# Metadata
metadata = {
    'title': DATASET_TITLE,
    'id': f'{KAGGLE_USER}/{DATASET_SLUG}',
    'licenses': [{'name': 'CC0-1.0'}],
}
(OUTPUT_DIR / 'dataset-metadata.json').write_text(
    json.dumps(metadata, indent=2), encoding='utf-8'
)

print('\nCriando dataset Kaggle...')
result = subprocess.run(
    ['kaggle', 'datasets', 'create', '-p', str(OUTPUT_DIR), '-r', 'zip'],
    capture_output=True, text=True,
)
print(result.stdout)
if result.returncode != 0:
    print('Create falhou, tentando version:')
    print(result.stderr)
    result = subprocess.run(
        ['kaggle', 'datasets', 'version', '-p', str(OUTPUT_DIR),
         '-m', 'Update from S5.8 notebook', '-r', 'zip'],
        capture_output=True, text=True,
    )
    print(result.stdout)
    if result.returncode != 0:
        print('ERRO:', result.stderr)
        raise RuntimeError('Falha ao publicar dataset.')

print(f'\nDataset URL: https://www.kaggle.com/datasets/{KAGGLE_USER}/{DATASET_SLUG}')

## Próximo passo: S5.9 — Análise estatística + docs/S5.md

Com os 3 CSVs publicados em `tcc-mri-s5-8-metrics`, o script `scripts/analyze_S5_9.py` (próxima sessão) fará:

### Testes estatísticos

| Teste | Métrica | Ref |
|---|---|---|
| **Friedman** + Nemenyi post-hoc | Coverage_lesion, IoU_topk_lesion, ULAS | Demšar (2006) |
| **Holm-Bonferroni** | Comparações par-a-par A-B, A-C, B-C | Holm (1979) |
| **BCa bootstrap** | IC 95% para cada métrica agregada (~47 vols) | Efron & Tibshirani (1993) |
| **Clopper-Pearson** | IC exato para Coverage como proporção | Clopper & Pearson (1934) |

### docs/S5.md — audiável para 3 audiências

1. **Você em 2 meses (pré-defesa)**: o que cada número significa e onde encontrar.
2. **Orientador**: visão completa, decisões-chave, riscos endereçados.
3. **Banca**: como reproduzir cada número (SHA-256 dos checkpoints + commit do código).

E o `tag` final da 4ª entrega.